In [ ]:
!pip install feedparser # 설치 : RSS에서 xml 태그별 정보 추출
!pip install newspaper3k # 설치 : 인터넷 신문 기사 분석을 위한 함수들(Article())
!pip install konlpy # 설치 : Korean Natural Language Py 한국어 형태소 분석기(명사 추출 목적)
!pip install lxml[html_clean] # 신규 주가(2024 가을)

import feedparser
from newspaper import Article
from konlpy.tag import Okt
from collections import Counter # 명사 추출 후 본문에 몇 번이나 그 명사가 나왔는지 확인(TF 구현용)
from bs4 import BeautifulSoup

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.3/81.3 kB 1.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6047 sha256=ad4ab8673a90d3c44af8fe9184d8a7f61f0c0d880026f05543a7e6be49bd06c7
  Stored in directory: /root/.cache/pip/wheels/f0/69/93/a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 46.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.9/104.9 kB 5.4 MB/s eta 0:00:00
  Created wheel for tinysegmenter: filename=tinysegmenter-0.3-py3-none-any.whl size=13539 sha256=8a9043468f7c94404142dbf54dcf47af9778cabaa3014c71bfcd414db51fc44b
  Stored in directory: /root/.cache/pip/wheels/c8/d6/6c/384f58df4

In [ ]:
# [단계 1] 모든 RSS파일(xml형식)을 돌아다니면서 기사의 제목과 링크를 추출
# urls는 우리가 검색할 RSS 파일들의 목록을 list로 만든 xml 파일들
urls = ["https://rss.etnews.com/Section901.xml",
        "https://rss.etnews.com/Section902.xml",
        "https://rss.etnews.com/Section903.xml",
        "https://rss.etnews.com/Section904.xml"]

In [ ]:
# 모든 RSS 안의 모든 기사들의 title과 link 추출
def crawl_rss(urls):
  array_rss = []
  titles_rss = set()
  for index_url in urls:
    print("[Crawl RSS]", index_url)
    parse_rss = feedparser.parse(index_url) # xml 파싱
    for p in parse_rss.entries:
      if p.title not in titles_rss:
        array_rss.append({'title':p.title, 'link':p.link})
        titles_rss.add(p.title)
      else:
        print("Duplicated ARticle:", p.title)
  return array_rss

In [ ]:
list_articles = crawl_rss(urls)

print(len(list_articles))
print(list_articles)

[Crawl RSS] https://rss.etnews.com/Section901.xml
[Crawl RSS] https://rss.etnews.com/Section902.xml
Duplicated ARticle: 위고페어, '원클릭 위조상품 신고서비스' 출시
[Crawl RSS] https://rss.etnews.com/Section903.xml
Duplicated ARticle: LG엔솔, 배터리SW 사업화…현대차와 사용료 협상
Duplicated ARticle: 외산 장비 설 곳 잃어가는 中 반도체 시장
Duplicated ARticle: 韓·美, 원자력수출협력 합의…산업부 “제2 웨스팅하우스 분쟁 막는다”
[Crawl RSS] https://rss.etnews.com/Section904.xml
Duplicated ARticle: 위고페어, '원클릭 위조상품 신고서비스' 출시
Duplicated ARticle: 쿠팡, 3분기 매출 10조6900억 '역대 최대'...고객도 11% 증가
Duplicated ARticle: '장 담그기' 유네스코 인류무형유산으로… “밥·김치 등 韓 문화의 핵심”
Duplicated ARticle: 막오른 `미국의 선택`...해리스-트럼프 초접전
Duplicated ARticle: 日 “공주는 안돼”… 유엔 '여성 왕위 계승' 권고 사실상 거부
Duplicated ARticle: 축구장에 날벼락…페루 선수 8명 쓰러졌다
Duplicated ARticle: 美 뉴욕서 6명 목숨 앗아간 '지하철 서핑'… 과연 뭐길래?
Duplicated ARticle: 짐 켈러·조주완 CEO 회동…텐스토렌트·LG전자 투자 협력 논의
Duplicated ARticle: '금투세 폐지', 이번달 처리 될까…'김건희 특검·이재명 1심' 막판 변수
Duplicated ARticle: 배달 상생협의체, 일부 단체 '요지부동'에 공회전…“수수료 5% 고수 비현실적” 지적
Duplicated ARticle: 임기 반환점 尹정부, 원전 계속운전 20년↑·상속세율

In [ ]:
# [단계 2] 모든 기사들의 본문 text 추출
# url을 입력받아서 그 안의 text 추출
def crawl_article(url, language='ko'):
  print("[Crawl Article]", url)
  a = Article(url, language=language)
  a.download() # url 기사 다운로드
  a.parse() # 분석 및 파싱
  return a.text # a.title

In [ ]:
for article in list_articles:
  text = crawl_article(article['link'])
  article['text'] = text

[Crawl Article] https://www.etnews.com/20241105000329
[Crawl Article] https://www.etnews.com/20241106000001
[Crawl Article] https://www.etnews.com/20241105000378
[Crawl Article] https://www.etnews.com/20241105000263
[Crawl Article] https://www.etnews.com/20241105000379
[Crawl Article] https://www.etnews.com/20241105000371
[Crawl Article] https://www.etnews.com/20241105000253
[Crawl Article] https://www.etnews.com/20241105000404
[Crawl Article] https://www.etnews.com/20241105000115
[Crawl Article] https://www.etnews.com/20241105000374
[Crawl Article] https://www.etnews.com/20241105000310
[Crawl Article] https://www.etnews.com/20241105000240
[Crawl Article] https://www.etnews.com/20241104000451
[Crawl Article] https://www.etnews.com/20241105000350
[Crawl Article] https://www.etnews.com/20241105000334
[Crawl Article] https://www.etnews.com/20241105000333
[Crawl Article] https://www.etnews.com/20241105000255
[Crawl Article] https://www.etnews.com/20241105000225
[Crawl Article] https://www.

In [ ]:
print(len(list_articles), list_articles[0])

76 {'title': "위고페어, '원클릭 위조상품 신고서비스' 출시", 'link': 'https://www.etnews.com/20241105000329', 'text': "상품 URL만 올리면 전 세계 온라인마켓 위조품 신고 끝\n\n인공지능(AI) 기반 위조상품 모니터링 및 차단 플랫폼을 운영하는 위고페어가 URL만 입력하면 간편하게 위조상품 신고할 수 있는 '원클릭 위조상품 신고 서비스'를 출시했다고 6일 밝혔다.\n\n\n\n'원클릭 위조상품 신고 서비스'는 위조품 판매 페이지 URL 입력만으로 위조상품 신고가 완료되어 전문지식이 없어도 누구나 쉽게 이용할 수 있다. 일명 '가품' 또는 '짝퉁'이라고 불리는 위조상품이 판매되는 것을 알고 있어도 플랫폼별로 상이한 신고 절차, 언어 장벽, 복잡한 신고 프로세스 등으로 대응을 못해 어려움을 겪던 기업들이 보다 쉽게 브랜드를 보호할 수 있도록 설계됐다. 특히 건별 신고 및 결제가 가능해 기업의 비용 부담을 대폭 줄인 것이 특징이다.\n\n\n\n위고페어 '원클릭 위조상품 신고'서비스는 국내뿐 아니라 다양한 글로벌 쇼핑몰에서 우리 기업의 브랜드 보호를 지원한다. 네이버, 쿠팡 등을 비롯해 알리바바, 테무, 쉬인 등 중국의 주요 이커머스 플랫폼은 물론, 쇼피, 라자다와 같은 동남아 플랫폼, 아마존, 이베이 등 글로벌 플랫폼의 위조상품까지 모두 신고가 가능하다.\n\n\n\n변리사와 지식재산보호원 위조상품 단속팀 출신 전문가들이 직접 신고 프로세스를 관리해 99%의 위조상품 차단 실적을 자랑한다. 기업별 전담 매니저를 배정해 맞춤형 서비스를 신속하게 제공한다.\n\n\n\n김종면 위고페어 CEO는 “위조상품 유통은 기업의 브랜드 이미지 훼손뿐만 아니라 매출 손실, 일자리 감소, 세수 감소 등 국가 경제에도 심각한 피해를 주고 있다”라며 “위고페어는 최첨단 AI 기술과 글로벌 플랫폼 위조품 차단 실무 경험을 보유한 전문 인력들을 바탕으로 우리 기업들의 브랜드를 효과적으로 보호하겠다”고 강조했다.\n\n\n\n위

In [ ]:
# [단계 3] 본문 text에서 명사 추출(키워드, 빈도)
def get_keywords(text, nKeywords=10):
  list_keywords = []

  spliter = Okt() # 문장을 형태소 별로 쪼개는 기능
  nouns = spliter.nouns(text) # 명사 추출
  count = Counter(nouns) # 추출된 명사의 출현 빈도수

  for n, c in count.most_common(nKeywords): # Counter.most_common : 출현 빈도 내림차순 keyword, count 리턴
    item = {'keyword':n, 'count':c}
    list_keywords.append(item)

  return list_keywords

In [ ]:
for article in list_articles:
  article['keywords'] = get_keywords(article['text'])

In [ ]:
print(list_articles[0]['keywords'])

[{'keyword': '상품', 'count': 14}, {'keyword': '신고', 'count': 13}, {'keyword': '위조', 'count': 13}, {'keyword': '위고', 'count': 7}, {'keyword': '페어', 'count': 7}, {'keyword': '서비스', 'count': 7}, {'keyword': '플랫폼', 'count': 6}, {'keyword': '기업', 'count': 6}, {'keyword': '수', 'count': 5}, {'keyword': '등', 'count': 5}]


In [ ]:
# [단계 4] 검색어를 입력받아서 그 검색어를 가지고 있는 문서를 출력
# 쿼리 입력 받아 문서의 keywords 리스트에 존재하는지 검색
def search_articles(query, list_keywords):
  nWords = 0
  for kw in list_keywords:
    if query == kw['keyword']:
      nWords = kw['count']
  return nWords

In [ ]:
query = input()

searched_articles = []

for article in list_articles:
  nQuery = search_articles(query, article['keywords'])
  if nQuery != 0:
    searched_articles.append((nQuery, article))

searched_articles.sort(key=lambda x: x[0], reverse=True)

for nQuery, article in searched_articles:
    print('[TF]', nQuery, '[Title]', article['title'])
    print('[URL]', article['link'])

상품
[TF] 14 [Title] 위고페어, '원클릭 위조상품 신고서비스' 출시
[URL] https://www.etnews.com/20241105000329
[TF] 7 [Title] 세븐일레븐, 업계 최초 '정온 푸드 운영 모델'…“데우지 않는 간편식”
[URL] https://www.etnews.com/20241106000007
[TF] 5 [Title] 인터파크 투어, 'W아이 겨울방학 영어캠프' 상품 출시
[URL] https://www.etnews.com/20241106000014
[TF] 4 [Title] 홈앤쇼핑, 이미용 특화 프로그램 '뷰티상담소' 인기
[URL] https://www.etnews.com/20241106000010
[TF] 4 [Title] 쿠팡, '블랙 프라이데이' 앞두고 가전·디지털 할인전…최대 75% 세일
[URL] https://www.etnews.com/20241104000290
